In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Exploratory Data Analysis
Let's load the data and get our bearings — how big is it, what's in it, and are there any obvious red flags before we start modelling.

In [ ]:
df = pd.read_csv('ml_features.csv')
df['Date'] = pd.to_datetime(df['Date'])

print(f"Dataset Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print("\nColumn Types:")
print(df.dtypes)

Missing Values & Class Imbalance

Fraud datasets are almost always heavily imbalanced. Let's see how bad it is here.

In [ ]:
null_counts = df.isnull().sum()
print("Missing values:")
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "None — dataset is fully complete!")

tx_imbalance = df['is_suspicious_tx'].value_counts()
tx_imbalance_pct = df['is_suspicious_tx'].value_counts(normalize=True) * 100
print("\nTransaction-Level Class Split:")
for val, count in tx_imbalance.items():
    print(f"  Class {val}: {count:6d} ({tx_imbalance_pct[val]:.3f}%)")

plt.figure(figsize=(6, 4))
sns.countplot(x='is_suspicious_tx', data=df, palette='viridis')
plt.title('Transaction-Level Target Imbalance')
plt.ylabel('Count')
plt.xlabel('Is Suspicious')
plt.show()

Temporal Patterns — Any Anomalies?
Always worth checking if suspicious activity clusters around certain dates. We noticed a spike in November — let's look at the October vs November split.

In [ ]:
daily_stats = df.groupby('Date').agg(
    total_transactions=('is_suspicious_tx', 'count'),
    suspicious_transactions=('is_suspicious_tx', 'sum')
)
daily_stats['fraud_rate_pct'] = (daily_stats['suspicious_transactions'] / daily_stats['total_transactions']) * 100
print(daily_stats)

df_oct = df[df['Date'].dt.month == 10]
df_nov = df[df['Date'].dt.month == 11]
print(f"\nOctober  | Transactions: {df_oct.shape[0]:6d} | Suspicious: {df_oct['is_suspicious_tx'].sum()} ({df_oct['is_suspicious_tx'].mean()*100:.3f}%)")
print(f"November | Transactions: {df_nov.shape[0]:6d} | Suspicious: {df_nov['is_suspicious_tx'].sum()} ({df_nov['is_suspicious_tx'].mean()*100:.1f}%)")

Transaction Amount Distribution
Amounts span many orders of magnitude, so we look at both raw and log-scale.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df['amount_local_npr'], bins=50, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Transaction Amount (NPR)')
axes[0].set_xlabel('Amount (Local NPR)')
axes[0].set_ylabel('Frequency')

sns.histplot(df['log_amount'], bins=50, kde=True, ax=axes[1], color='navy')
axes[1].set_title('Log(Transaction Amount)')
axes[1].set_xlabel('Log Amount')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

Feature Correlations with Fraud Label

A quick look at which raw features already correlate with suspicious transactions before we do any engineering.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['Sender_account', 'Receiver_account']]

correlations = df[numeric_cols].corr()['is_suspicious_tx'].sort_values()

plt.figure(figsize=(10, 8))
sns.barplot(x=correlations.values, y=correlations.index, palette='coolwarm')
plt.title('Feature Correlation with is_suspicious_tx')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Features')
plt.show()